In [0]:
# Gold: Train Churn Prediction Model
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import pandas as pd

df_gold_source = spark.table("silver_customer_churn").toPandas()

feature_cols = ['tenure_months', 'monthly_charges', 'support_tickets',
                 'contract_type_encoded', 'payment_method_encoded',
                 'internet_service_encoded', 'has_streaming_encoded', 'spend_per_tenure']

X = df_gold_source[feature_cols]
y = df_gold_source['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall: {recall_score(y_test, y_pred):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.3f}")

# Feature importance
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

display(importance_df)

Accuracy: 0.823
Precision: 0.815
Recall: 0.493
ROC-AUC: 0.878


feature,importance
contract_type_encoded,0.4473994320412509
support_tickets,0.21179743402301318
spend_per_tenure,0.09292087883772998
payment_method_encoded,0.07601718852304656
tenure_months,0.07540581053506487
monthly_charges,0.07052014038412811
internet_service_encoded,0.01679774371460748
has_streaming_encoded,0.009141371941159078


In [0]:
# Gold: Write churn predictions and feature importance to Delta

# Build full prediction set (train + test combined for full customer scoring)
df_gold_source['churn_probability'] = model.predict_proba(df_gold_source[feature_cols])[:, 1]
df_gold_source['churn_predicted'] = model.predict(df_gold_source[feature_cols])

# Risk segments for dashboard
df_gold_source['risk_segment'] = pd.cut(
    df_gold_source['churn_probability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

gold_predictions = df_gold_source[['customer_id', 'tenure_months', 'contract_type',
                                     'monthly_charges', 'support_tickets', 'payment_method',
                                     'churned', 'churn_probability', 'churn_predicted', 'risk_segment']]

# Write predictions table
spark.createDataFrame(gold_predictions).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_churn_predictions")

# Write feature importance table
spark.createDataFrame(importance_df).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_feature_importance")

# Write model performance metrics
metrics_df = pd.DataFrame({
    'metric': ['Accuracy', 'Precision', 'Recall', 'ROC-AUC'],
    'value': [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),
              recall_score(y_test, y_pred), roc_auc_score(y_test, y_pred_proba)]
})
spark.createDataFrame(metrics_df).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_model_metrics")

print("Gold tables written: gold_churn_predictions, gold_feature_importance, gold_model_metrics")

Gold tables written: gold_churn_predictions, gold_feature_importance, gold_model_metrics


In [0]:
# Add residual/calibration data for predicted vs actual visual
import pandas as pd

# Read existing predictions
df_viz = spark.table("gold_churn_predictions").toPandas()

# Bin predicted probabilities into deciles for a calibration-style view
df_viz['probability_bucket'] = pd.cut(df_viz['churn_probability'], bins=10, labels=False)

calibration = df_viz.groupby('probability_bucket').agg(
    avg_predicted_prob=('churn_probability', 'mean'),
    actual_churn_rate=('churned', 'mean'),
    customer_count=('customer_id', 'count')
).reset_index()

spark.createDataFrame(calibration).write \
    .format("delta").mode("overwrite") \
    .saveAsTable("gold_model_calibration")

print(f"Calibration table written: {len(calibration)} probability buckets")
display(calibration)

Calibration table written: 10 probability buckets


probability_bucket,avg_predicted_prob,actual_churn_rate,customer_count
0,0.03381916998807227,0.0011248593925759281,1778
1,0.14313567641708563,0.06319702602230483,269
2,0.2699865769370062,0.13246753246753246,770
3,0.323135149937139,0.3592880978865406,899
4,0.45645590444675876,0.4557823129251701,294
5,0.5211137460915112,0.7514970059880239,334
6,0.6317405169025584,0.582089552238806,67
7,0.742543076151674,0.8709677419354839,155
8,0.8234443193903114,0.9803370786516854,356
9,0.9006069972494044,0.9871794871794872,78
